# DIT5411 — Traditional Chinese Handwriting Recognition
Works in **Google Colab** (T4 GPU) or **local Jupyter**. In Colab, enable GPU: Runtime → Change runtime type → T4 GPU. In Jupyter, select the project's `.venv` kernel.

In [ ]:
import os, sys, subprocess

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    if not os.path.isfile('train.py'):
        if not os.path.isdir('DIT5411-LuiYuHin'):
            subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
        os.chdir('DIT5411-LuiYuHin')
    subprocess.run(['git', 'pull', '-q'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
else:
    if not os.path.isfile('train.py'):
        os.chdir('..')   # notebooks/ -> project root
    print('Local Jupyter mode — using project .venv kernel.')

print('Working directory:', os.getcwd())

# Always train with the project venv (it has TensorFlow), not the Jupyter kernel.
PYTHON = sys.executable
if not IN_COLAB:
    venv_py = os.path.join(os.getcwd(), '.venv', 'Scripts', 'python.exe')
    if os.path.isfile(venv_py):
        PYTHON = venv_py
print('Training Python:', PYTHON)

## Dataset check
Local project already has `data/cleaned_data` extracted. In Colab run the download cell below once.

In [ ]:
d = 'data/cleaned_data'
if os.path.isdir(d):
    print(f'Dataset ready: {len(os.listdir(d))} character folders')
else:
    print('Dataset missing — run the Colab download cell below, or extract_dataset.py locally.')

In [ ]:
# Colab only: download + extract the 13,065-character dataset (~1.1 GB zip, ~15 min)
if IN_COLAB and not os.path.isdir('data/cleaned_data'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    import glob
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py'])
    print('Done:', len(os.listdir('data/cleaned_data')), 'folders')

## Quick pilot (recommended first)
20 characters, 2 epochs — confirms the pipeline works. Local CPU is fine for this.

In [ ]:
r = subprocess.run([PYTHON, 'train.py', '--num-classes', '20', '--epochs', '2',
                    '--models', 'A', 'B', '--no-imagenet',
                    '--steps-per-epoch', '100', '--output-dir', 'results_pilot'])
print('exit code:', r.returncode)

## Full assignment run
Use **Colab GPU** — local CPU would take days. Each epoch is capped at 3000 steps to fit Colab's free session limit.

In [ ]:
r = subprocess.run([PYTHON, 'train.py', '--epochs', '5',
                    '--steps-per-epoch', '3000', '--models', 'A', 'B', 'C', '--resume'])
print('exit code:', r.returncode)

If the runtime disconnects, re-run the cell above: `--resume` continues from the last saved epoch.

In [ ]:
from IPython.display import Markdown, display
p = 'results/RESULTS.md'
display(Markdown(open(p, encoding='utf-8').read()) if os.path.isfile(p) else 'RESULTS.md not generated yet — finish training first.')